In [1]:
import glob

import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from utils.dataloader import load_shards

local_pattern = '../data/koppen_shard_part_*.tfrecord.gz'
all_files = glob.glob(local_pattern)

if len(all_files) > 0:
    print(f"Found {len(all_files)} shards in local cache.")
else:
    raise FileNotFoundError("Data not found. Please download the dataset using the OneDrive link in the README.")

Found 1500 shards in local cache.


In [2]:
train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42)
print(f"Training on {len(train_files)} shards.")
print(f"Testing on {len(test_files)} shards.")

Training on 1200 shards.
Testing on 300 shards.


In [3]:
scaler = StandardScaler()
all_classes = np.arange(1, 31)
batch_size = 128
train_dataset = load_shards(train_files, batch_size=batch_size)

# --- Pass 1: Calculating scaling statistics ---
print("Pass 1: Calculating scaling statistics...")
for X_batch, _ in train_dataset.as_numpy_iterator():
    X_flat = X_batch.reshape(X_batch.shape[0], -1)
    scaler.partial_fit(X_flat)

print("Scaling statistics computed")

Pass 1: Calculating scaling statistics...
Scaling statistics computed


In [4]:
# Use SGDClassifier with loss='log_loss' to implement Logistic Regression with partial_fit
clf = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    alpha=10,
    random_state=42
)

# --- Pass 2: Model training ---
print('Pass 2: Training model...')
for i, (X_batch, y_batch) in enumerate(train_dataset.as_numpy_iterator()):
    X_flat = X_batch.reshape(X_batch.shape[0], -1)

    # Apply scaling
    X_scaled = scaler.transform(X_flat)

    # Fit model on current batch
    clf.partial_fit(X_scaled, y_batch, classes=all_classes)

    if i % 20 == 0:
        print(f"Trained on batch {i}")

print("Training completed")

Pass 2: Training model...
Trained on batch 0
Trained on batch 20
Trained on batch 40
Trained on batch 60
Trained on batch 80
Trained on batch 100
Trained on batch 120
Trained on batch 140
Trained on batch 160
Trained on batch 180
Trained on batch 200
Trained on batch 220
Trained on batch 240
Trained on batch 260
Trained on batch 280
Trained on batch 300
Trained on batch 320
Trained on batch 340
Trained on batch 360
Training completed


In [5]:
# --- Evaluation ---
print("Evaluating on training set...")
y_train_true, y_train_pred = [], []

for X_batch, y_batch in train_dataset.as_numpy_iterator():
    X_flat = X_batch.reshape(X_batch.shape[0], -1)
    X_scaled = scaler.transform(X_flat)

    y_train_pred.extend(clf.predict(X_scaled))
    y_train_true.extend(y_batch)

print("Evaluating on test set...")
test_ds = load_shards(test_files, batch_size=batch_size)
y_test_true, y_test_pred = [], []

for X_batch, y_batch in test_ds.as_numpy_iterator():
    X_flat = X_batch.reshape(X_batch.shape[0], -1)
    X_scaled = scaler.transform(X_flat)

    y_test_pred.extend(clf.predict(X_scaled))
    y_test_true.extend(y_batch)

Evaluating on training set...
Evaluating on test set...


In [6]:
print("Training classification report:")
print(classification_report(y_train_true, y_train_pred, zero_division=0))
print('\n\n')
print("Test classification report:")
print(classification_report(y_test_true, y_test_pred, zero_division=0))

Training classification report:
              precision    recall  f1-score   support

           1       0.42      0.77      0.55      1292
           2       0.56      0.01      0.03      1540
           3       0.60      0.00      0.01      1580
           4       0.30      0.80      0.44      1607
           5       0.41      0.27      0.33      1583
           6       0.25      0.05      0.08      1599
           7       0.00      0.00      0.00      1586
           8       0.00      0.00      0.00      1616
           9       0.04      0.73      0.08      1605
          10       0.00      0.00      0.00      1592
          11       0.00      0.00      0.00      1598
          12       0.00      0.00      0.00      1582
          13       0.76      0.03      0.06      1576
          14       0.00      0.00      0.00      1647
          15       0.15      0.50      0.23      1485
          16       0.22      0.24      0.23      1622
          17       0.00      0.00      0.00      